In [1]:
import re
import string
import warnings
from pathlib import Path

import nltk
import pandas as pd

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
nltk.download("stopwords")
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\dan2s\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\dan2s\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\dan2s\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

print("NLTK resources loaded.")

NLTK resources loaded.


In [4]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

DATA_PATH = PROJECT_ROOT / "data" / "spam_eda.csv"

df = pd.read_csv(DATA_PATH)

df.head()

,label,message,Unnamed: 2,Unnamed: 3,Unnamed: 4,message_length,word_count
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN,111,20
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN,29,6
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN,155,28
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN,49,11
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN,61,13


In [5]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 5169 entries, 0 to 5168
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   label           5169 non-null   str  
 1   message         5169 non-null   str  
 2   Unnamed: 2      43 non-null     str  
 3   Unnamed: 3      10 non-null     str  
 4   Unnamed: 4      5 non-null      str  
 5   message_length  5169 non-null   int64
 6   word_count      5169 non-null   int64
dtypes: int64(2), str(5)
memory usage: 703.0 KB
None


In [6]:
df = df[["label", "message"]].copy()

df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [7]:
df["clean_message"] = df["message"].str.lower()

df.head()

,label,message,clean_message
0,ham,"Go until jurong point, crazy.. Available only ...","go until jurong point, crazy.. available only ..."
1,ham,Ok lar... Joking wif u oni...,ok lar... joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in 2 a wkly comp to win fa cup fina...
3,ham,U dun say so early hor... U c already then say...,u dun say so early hor... u c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro...","nah i don't think he goes to usf, he lives aro..."


In [8]:
df["clean_message"] = df["clean_message"].apply(
    lambda text: re.sub(r"http\S+|www\S+|https\S+", "", text)
)

In [9]:
df["clean_message"] = df["clean_message"].apply(
    lambda text: re.sub(r"\S+@\S+", "", text)
)

In [10]:
df["clean_message"] = df["clean_message"].apply(
    lambda text: re.sub(r"\d{7,}", "", text)
)

In [11]:
df["clean_message"] = df["clean_message"].apply(
    lambda text: re.sub(r"\d+", " ", text)
)

In [12]:
translator = str.maketrans("", "", string.punctuation)

df["clean_message"] = df["clean_message"].apply(
    lambda text: text.translate(translator)
)

In [13]:
df["clean_message"] = df["clean_message"].apply(
    lambda text: " ".join(text.split())
)

In [14]:
df["tokens"] = df["clean_message"].apply(word_tokenize)

df.head()

,label,message,clean_message,tokens
0,ham,"Go until jurong point, crazy.. Available only ...",go until jurong point crazy available only in ...,"[go, until, jurong, point, crazy, available, o..."
1,ham,Ok lar... Joking wif u oni...,ok lar joking wif u oni,"[ok, lar, joking, wif, u, oni]"
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in a wkly comp to win fa cup final ...,"[free, entry, in, a, wkly, comp, to, win, fa, ..."
3,ham,U dun say so early hor... U c already then say...,u dun say so early hor u c already then say,"[u, dun, say, so, early, hor, u, c, already, t..."
4,ham,"Nah I don't think he goes to usf, he lives aro...",nah i dont think he goes to usf he lives aroun...,"[nah, i, dont, think, he, goes, to, usf, he, l..."


In [15]:
df["tokens"] = df["tokens"].apply(
    lambda words: [
        word
        for word in words
        if word not in stop_words
    ]
)

In [16]:
df["tokens"] = df["tokens"].apply(
    lambda words: [
        stemmer.stem(word)
        for word in words
    ]
)

In [17]:
df["processed_message"] = df["tokens"].apply(
    lambda words: " ".join(words)
)

df.head()

,label,message,clean_message,tokens,processed_message
0,ham,"Go until jurong point, crazy.. Available only ...",go until jurong point crazy available only in ...,"[go, jurong, point, crazi, avail, bugi, n, gre...",go jurong point crazi avail bugi n great world...
1,ham,Ok lar... Joking wif u oni...,ok lar joking wif u oni,"[ok, lar, joke, wif, u, oni]",ok lar joke wif u oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in a wkly comp to win fa cup final ...,"[free, entri, wkli, comp, win, fa, cup, final,...",free entri wkli comp win fa cup final tkt st m...
3,ham,U dun say so early hor... U c already then say...,u dun say so early hor u c already then say,"[u, dun, say, earli, hor, u, c, alreadi, say]",u dun say earli hor u c alreadi say
4,ham,"Nah I don't think he goes to usf, he lives aro...",nah i dont think he goes to usf he lives aroun...,"[nah, dont, think, goe, usf, live, around, tho...",nah dont think goe usf live around though


In [18]:
empty_messages = (
    df["processed_message"]
    .str.strip()
    .eq("")
    .sum()
)

print(f"Empty processed messages: {empty_messages}")

Empty processed messages: 6


In [19]:
df = df[
    df["processed_message"].str.strip() != ""
].reset_index(drop=True)

print(df.shape)

(5163, 5)


In [20]:
label_map = {
    "ham": 0,
    "spam": 1
}

df["target"] = df["label"].map(label_map)

df.head()

,label,message,clean_message,tokens,processed_message,target
0,ham,"Go until jurong point, crazy.. Available only ...",go until jurong point crazy available only in ...,"[go, jurong, point, crazi, avail, bugi, n, gre...",go jurong point crazi avail bugi n great world...,0
1,ham,Ok lar... Joking wif u oni...,ok lar joking wif u oni,"[ok, lar, joke, wif, u, oni]",ok lar joke wif u oni,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in a wkly comp to win fa cup final ...,"[free, entri, wkli, comp, win, fa, cup, final,...",free entri wkli comp win fa cup final tkt st m...,1
3,ham,U dun say so early hor... U c already then say...,u dun say so early hor u c already then say,"[u, dun, say, earli, hor, u, c, alreadi, say]",u dun say earli hor u c alreadi say,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",nah i dont think he goes to usf he lives aroun...,"[nah, dont, think, goe, usf, live, around, tho...",nah dont think goe usf live around though,0


In [21]:
print(df["target"].value_counts())

target
0    4510
1     653
Name: count, dtype: int64


In [22]:
df[
    [
        "label",
        "target",
        "processed_message"
    ]
].head()

,label,target,processed_message
0,ham,0,go jurong point crazi avail bugi n great world...
1,ham,0,ok lar joke wif u oni
2,spam,1,free entri wkli comp win fa cup final tkt st m...
3,ham,0,u dun say earli hor u c alreadi say
4,ham,0,nah dont think goe usf live around though


In [23]:
OUTPUT_PATH = PROJECT_ROOT / "data" / "spam_processed.csv"

df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Processed dataset saved successfully.")
print(OUTPUT_PATH)

Processed dataset saved successfully.
c:\Users\dan2s\OneDrive\Documents\ONE\Desktop\Fraud_SMS_Classifier\data\spam_processed.csv


In [24]:
print("=" * 60)
print("TEXT PREPROCESSING SUMMARY")
print("=" * 60)

print(f"Total Messages : {len(df)}")
print(f"Ham Messages   : {(df['target']==0).sum()}")
print(f"Spam Messages  : {(df['target']==1).sum()}")

print(f"\nColumns:")
print(df.columns.tolist())

print("\nSample Processed Message:")
print(df.loc[0, "processed_message"])

print("=" * 60)
print("TEXT PREPROCESSING COMPLETED")
print("=" * 60)

TEXT PREPROCESSING SUMMARY
Total Messages : 5163
Ham Messages   : 4510
Spam Messages  : 653

Columns:
['label', 'message', 'clean_message', 'tokens', 'processed_message', 'target']

Sample Processed Message:
go jurong point crazi avail bugi n great world la e buffet cine got amor wat
TEXT PREPROCESSING COMPLETED
